# 指数衰减突触电流与恒定电流对照

Homework 2 · Section 1 · Part 4 · Option A。基础模型见 [LIF 教程](neuron_dynamics.ipynb)，阈值实验见 [阈值对照](threshold_effect/threshold_effect.ipynb)。本实验仅比较输入时间分布的影响。

在项目根目录准备 `uv sync --locked --extra cpu --dev`，Notebook 选择同一环境内核。


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

root = Path.cwd() if (Path.cwd() / "lif").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from lif.model import LIFNeuron, SimulationParameters, firing_rate, simulate_current  # noqa: E402
from lif.plot_lif import plot_summary  # noqa: E402

p = SimulationParameters()
out = root / "lif" / "synaptic_current_results"
out.mkdir(parents=True, exist_ok=True)

## 输入与控制变量

每 50 ms 从 20 ms 开始产生一次事件，每次增加 20 nA，按时间常数 5 ms 指数衰减：

$$I(t)=\sum_{t_j\le t}20\exp[-(t-t_j)/5]\;\mathrm{nA}.$$

恒定输入取有限仿真窗口内突触电流的平均值。两组保持神经元参数和仿真时长相同；峰值与时间分布不同，因此相同平均电流不要求相同发放率。


In [ ]:
input_time = np.arange(round(p.duration / p.dt)) * p.dt
synaptic = np.zeros_like(input_time)
for event_time in np.arange(20.0, p.duration, 50.0):
    elapsed = input_time - event_time
    active = elapsed >= 0
    synaptic[active] += 20.0 * np.exp(-elapsed[active] / 5.0)
currents = {
    "Constant current": np.full_like(input_time, synaptic.mean()),
    "Synaptic current": synaptic,
}
assert np.isclose(currents["Constant current"].mean(), synaptic.mean())

In [ ]:
import csv
import json
from dataclasses import asdict

rows = []
traces = {}
for name, current in currents.items():
    neuron = LIFNeuron(p.dt, p.tau, p.resistance, p.rest, p.threshold, p.refractory)
    with torch.inference_mode():
        voltage, spikes = simulate_current(current, p.duration, p.dt, neuron)
    times = input_time + p.dt
    data = {
        "input_time_ms": input_time,
        "time_ms": times,
        "current": current,
        "voltage": voltage,
        "spikes": spikes,
        "rate": firing_rate(spikes, p.dt),
    }
    traces[name] = data
    plot_summary(name, data, p.rest, p.threshold)
    plt.gcf().savefig(out / (name.lower().replace(" ", "_") + ".png"), dpi=150)
    rows.append(
        {
            "input": name,
            "mean_current_nA": float(current.mean()),
            "spike_count": int(spikes.sum()),
            "average_rate_Hz": float(spikes.sum() / (p.duration / 1000)),
        }
    )
with (out / "summary.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0])
    writer.writeheader()
    writer.writerows(rows)
(out / "parameters.json").write_text(json.dumps(asdict(p), indent=2), encoding="utf-8")
for row in rows:
    print(row)

## 如何解释结果

结合曲线比较膜电位的持续积累和事件后的短时响应，使用完整 2 s 的脉冲数与平均发放率判断差异。四联图只展示前 200 ms。泄漏、阈值和不应期使输入到脉冲的变换非线性，因此不能只比较平均电流。输出在 `lif/synaptic_current_results/`；统计结果由运行生成。
